# 호텔 예약 취소율 분석 — 정책·UX 개선 가설 검증

## 분석 배경

호텔 비즈니스에서 **예약 취소**는 직접적인 매출 손실이자 운영 효율 저하의 원인이다.
빈 객실은 마지막 순간까지 재판매가 어렵고, 인력·식음료·청소 등 가변 비용은 이미 발생한다.

본 분석은 약 12만 건의 실제 호텔 예약 데이터(2015~2017, Resort + City)를 대상으로
**어떤 신호가 취소를 예측하는가**, 그리고 그 신호를 기반으로
**어떤 정책·UX 개선을 기획할 수 있는가**를 도출한다.

> **데이터 출처**: Kaggle "Hotel Booking Demand" (Antonio, Almeida & Nunes, 2019)  
> **타겟 변수**: `is_canceled` (0/1)  
> **분석자 관점**: 서비스 기획자 — 데이터로 가설을 검증하고 의사결정으로 연결

## 풀고자 하는 질문

> **"취소율을 줄이려면 어디에, 무엇을 손대야 하는가?"**

답을 찾기 위해 다음 6개 가설을 데이터로 검증한다.

| # | 가설 | 검증 변수 | 기대 인사이트 |
|---|---|---|---|
| **H1** | 예약 lead time(예약~체크인 간격)이 길수록 취소율이 높다 | `lead_time` × `is_canceled` | 장기 예약자 대상 정책 설계 |
| **H2** | 보증금 없는 예약(No Deposit)이 압도적으로 취소율이 높다 | `deposit_type` × `is_canceled` | 보증금 정책의 효과 정량화 |
| **H3** | 이전에 취소 이력이 있는 고객은 다시 취소할 위험이 크다 | `previous_cancellations` × `is_canceled` | 상습 취소자 식별·관리 |
| **H4** | 마켓 세그먼트별 취소율이 다르다 (특히 Groups, Online TA) | `market_segment` × `is_canceled` | 채널별 차별화 정책 |
| **H5** | 예약 변경 횟수가 많은 예약은 취소율이 다르다 | `booking_changes` × `is_canceled` | 운영 마찰 신호 발굴 |
| **H6** | 월별(시즌별) 취소율 차이가 있다 | `arrival_date_month` × `is_canceled` | 성수기·비수기 차별 운영 |

## 분석 절차

1. **데이터 로드 & 검수** — 결측·중복·이상치 확인
2. **EDA** — 타겟 분포, 호텔별·연도별 베이스라인 취소율
3. **가설 검증 (H1~H6)** — 각 가설별로:
   - 시각화 (그룹별 취소율 비교)
   - 그룹 간 차이의 통계적 유의성 확인
   - **신뢰구간**으로 차이의 크기와 불확실성을 함께 표시
4. **종합 인사이트** — 가설별 검증 결과 요약
5. **기획 권고** — 정책 / UX / 운영 측면에서 적용 가능한 개선안 도출

### 사용할 통계 도구 (개념만)

- **그룹 비교**: 두 그룹 취소율 차이의 **95% 신뢰구간**  
  → "차이의 띠가 0을 포함하면 차이 없음, 0을 안 넘기면 유의미한 차이"
- **카테고리 변수 검정**: 카이제곱 독립성 검정  
  → "두 변수의 관계가 우연일 가능성이 얼마나 적은가"

> 통계 자체가 목적이 아니라 **기획적 의사결정의 근거**를 만드는 도구로 사용한다.
> 숫자 하나로 단정 짓지 않고, **신뢰구간 폭**으로 불확실성도 함께 보여준다.

## 데이터의 한계 — 먼저 짚고 가기

분석 결론을 적용할 때 다음 조건들을 함께 고려한다:

1. **도메인 한정**: 포르투갈 소재 호텔 2개 (Resort + City). 한국·아시아 시장 일반화는 신중
2. **시점 한정**: 2015~2017년 (코로나 이전). 팬데믹 이후 패턴 변화 미반영
3. **인과관계 ≠ 상관관계**: 관측 데이터 기반 → 정책 변경의 **인과적** 효과는 별도 A/B 검증 필요
4. **타겟 정의**: `is_canceled = 1`은 체크인 전 취소만 포함 (No-Show는 별도 status)

> 이 한계들은 결론부에서 **"권고 적용 전 추가로 필요한 검증"**으로 다시 정리한다.

In [36]:
import warnings
warnings.filterwarnings('ignore')

import pandas as pd
import numpy as np
from scipy import stats
from statsmodels.stats.proportion import proportion_confint, confint_proportions_2indep

import matplotlib.pyplot as plt
import seaborn as sns

pd.set_option('display.max_columns', 50)
pd.set_option('display.width', 200)

sns.set_style('whitegrid')
plt.rcParams['figure.dpi'] = 100
plt.rcParams['font.family'] = 'Malgun Gothic'
plt.rcParams['axes.unicode_minus'] = False

In [37]:
df = pd.read_csv('data/hotel_bookings.csv')
print(f'행: {df.shape[0]:,} / 열: {df.shape[1]}')
df.head()

행: 119,390 / 열: 32


,hotel,is_canceled,lead_time,arrival_date_year,arrival_date_month,arrival_date_week_number,arrival_date_day_of_month,stays_in_weekend_nights,stays_in_week_nights,adults,children,babies,meal,country,market_segment,distribution_channel,is_repeated_guest,previous_cancellations,previous_bookings_not_canceled,reserved_room_type,assigned_room_type,booking_changes,deposit_type,agent,company,days_in_waiting_list,customer_type,adr,required_car_parking_spaces,total_of_special_requests,reservation_status,reservation_status_date
0,Resort Hotel,0,342,2015,July,27,1,0,0,2,0.0,0,BB,PRT,Direct,Direct,0,0,0,C,C,3,No Deposit,NaN,NaN,0,Transient,0.0,0,0,Check-Out,2015-07-01
1,Resort Hotel,0,737,2015,July,27,1,0,0,2,0.0,0,BB,PRT,Direct,Direct,0,0,0,C,C,4,No Deposit,NaN,NaN,0,Transient,0.0,0,0,Check-Out,2015-07-01
2,Resort Hotel,0,7,2015,July,27,1,0,1,1,0.0,0,BB,GBR,Direct,Direct,0,0,0,A,C,0,No Deposit,NaN,NaN,0,Transient,75.0,0,0,Check-Out,2015-07-02
3,Resort Hotel,0,13,2015,July,27,1,0,1,1,0.0,0,BB,GBR,Corporate,Corporate,0,0,0,A,A,0,No Deposit,304.0,NaN,0,Transient,75.0,0,0,Check-Out,2015-07-02
4,Resort Hotel,0,14,2015,July,27,1,0,2,2,0.0,0,BB,GBR,Online TA,TA/TO,0,0,0,A,A,0,No Deposit,240.0,NaN,0,Transient,98.0,0,1,Check-Out,2015-07-03


---

## 1. EDA — 데이터 검수 + 베이스라인

분석에 들어가기 전 데이터 품질을 확인하고, 전체 취소율의 베이스라인을 잡는다.

In [38]:
na = df.isna().sum()
na = na[na > 0].sort_values(ascending=False)
print(['결측치가 있는 컬럼'])
print(na)

# 중복
dup = df.duplicated().sum()
print(f'\n[중복 행] {dup:,}건 ({dup/len(df)*100:.1f}%)')

# adr (1박 평균요금) 이상치
print(f'\n[adr 이상치]')
print(f"  음수:    {(df['adr'] < 0).sum()}건")
print(f"  >1000:  {(df['adr'] > 1000).sum()}건")
print(f"  =0:     {(df['adr'] == 0).sum()}건")


['결측치가 있는 컬럼']
company     112593
agent        16340
country        488
children         4
dtype: int64

[중복 행] 31,994건 (26.8%)

[adr 이상치]
  음수:    1건
  >1000:  1건
  =0:     1959건


In [44]:
df_tmp = df[df.duplicated(keep=False)].sort_values(list(df.columns))
print(f'중복 묶음에 속한 행: {len(df_tmp):,}건')
df_tmp.head(20)


중복 묶음에 속한 행: 40,165건


,hotel,is_canceled,lead_time,arrival_date_year,arrival_date_month,arrival_date_week_number,arrival_date_day_of_month,stays_in_weekend_nights,stays_in_week_nights,adults,children,babies,meal,country,market_segment,distribution_channel,is_repeated_guest,previous_cancellations,previous_bookings_not_canceled,reserved_room_type,assigned_room_type,booking_changes,deposit_type,agent,company,days_in_waiting_list,customer_type,adr,required_car_parking_spaces,total_of_special_requests,reservation_status,reservation_status_date
40772,City Hotel,0,0,2015,August,32,7,0,2,2,0.0,0,BB,PRT,Direct,Direct,0,0,0,A,A,0,No Deposit,14.0,NaN,0,Transient,75.0,0,1,Check-Out,2015-08-09
40802,City Hotel,0,0,2015,August,32,7,0,2,2,0.0,0,BB,PRT,Direct,Direct,0,0,0,A,A,0,No Deposit,14.0,NaN,0,Transient,75.0,0,1,Check-Out,2015-08-09
40821,City Hotel,0,0,2015,August,32,8,0,1,2,0.0,0,SC,PRT,Online TA,TA/TO,0,0,0,A,A,0,No Deposit,9.0,NaN,0,Transient,89.0,0,1,Check-Out,2015-08-09
40838,City Hotel,0,0,2015,August,32,8,0,1,2,0.0,0,SC,PRT,Online TA,TA/TO,0,0,0,A,A,0,No Deposit,9.0,NaN,0,Transient,89.0,0,1,Check-Out,2015-08-09
76792,City Hotel,0,0,2015,August,33,10,1,0,2,0.0,0,SC,FRA,Online TA,TA/TO,0,0,0,A,A,0,No Deposit,NaN,NaN,0,Transient,75.0,0,0,Check-Out,2015-08-11
76793,City Hotel,0,0,2015,August,33,10,1,0,2,0.0,0,SC,FRA,Online TA,TA/TO,0,0,0,A,A,0,No Deposit,NaN,NaN,0,Transient,75.0,0,0,Check-Out,2015-08-11
76794,City Hotel,0,0,2015,August,33,10,1,0,2,0.0,0,SC,FRA,Online TA,TA/TO,0,0,0,A,A,0,No Deposit,NaN,NaN,0,Transient,75.0,0,0,Check-Out,2015-08-11
41067,City Hotel,0,0,2015,August,33,11,0,1,1,0.0,0,BB,PRT,Corporate,Corporate,0,0,0,A,A,0,No Deposit,NaN,38.0,0,Transient-Party,88.0,0,0,Check-Out,2015-08-12
41073,City Hotel,0,0,2015,August,33,11,0,1,1,0.0,0,BB,PRT,Corporate,Corporate,0,0,0,A,A,0,No Deposit,NaN,38.0,0,Transient-Party,88.0,0,0,Check-Out,2015-08-12
41071,City Hotel,0,0,2015,August,33,11,0,1,2,0.0,0,BB,PRT,Direct,Direct,0,0,0,A,A,0,No Deposit,NaN,NaN,0,Transient,80.0,0,0,Check-Out,2015-08-12


### 결측치

| 이슈 | 규모 | 처리 방향 |
|---|---|---|
| `company` 결측 | **112,593건 (94.3%)** | 컬럼 자체를 분석에서 제외 또는 "법인예약 여부" 0/1 변환 |
| `agent` 결측 | 16,340건 (13.7%) | 중개사 사용 여부 변환 가능 |
| `country` 결측 | 488건 (0.4%) | 그대로 사용 |
| **중복 행** | **31,994건 (26.8%)** | 약 1/4가 중복 — 단순 중복인지, 의도된 동일 조건 다중 행인지 별도 확인 필요 |
| `adr` 이상치 | 음수 1, 0인 행 1,959, >1000 1 | 본 분석은 `is_canceled` 중심이라 `adr` 미사용 |

### 중복 행 처리 — 가정 기반 접근

**관찰된 사실**
- 전체 119,390건 중 31,994건 (26.8%)이 모든 컬럼이 동일
- 중복 묶음의 DataFrame index는 비연속적 (예: 40772, 40802, 40821 …)

**한계 — 진실을 알 수 없는 이유**
- 데이터셋에 **호텔명·객실 호수·예약 timestamp, booking_id** 같은 고유 식별자가 없음
- 따라서 동일해 보이는 두 행이 실제로
  - (A) 같은 예약을 시스템이 중복 적재한 것인지
  - (B) 서로 다른 손님이 같은 날·같은 룸타입을 우연히 따로 예약한 것인지
- **데이터만으로는 구분 불가능**

**우리의 가정**
> 식별자 부재로 인한 표면적 중복일 가능성이 더 크다고 보고,
> **모든 행을 별개 예약으로 취급한다.**
>
> - 만약 (A) 시나리오가 맞다면 표본 크기가 부풀려져 신뢰구간이 실제보다 좁게 나옴 → 결과 해석 시 보수적으로 받아들임
> - 추적 편의를 위해 `booking_id = index + 1` 부여

In [47]:
df['booking_id'] = df.index + 1
print(f'booking_id 부여: {df['booking_id'].nunique()} 건')
df.head()

booking_id 부여: 119390 건


,hotel,is_canceled,lead_time,arrival_date_year,arrival_date_month,arrival_date_week_number,arrival_date_day_of_month,stays_in_weekend_nights,stays_in_week_nights,adults,children,babies,meal,country,market_segment,distribution_channel,is_repeated_guest,previous_cancellations,previous_bookings_not_canceled,reserved_room_type,assigned_room_type,booking_changes,deposit_type,agent,company,days_in_waiting_list,customer_type,adr,required_car_parking_spaces,total_of_special_requests,reservation_status,reservation_status_date,booking_id
0,Resort Hotel,0,342,2015,July,27,1,0,0,2,0.0,0,BB,PRT,Direct,Direct,0,0,0,C,C,3,No Deposit,NaN,NaN,0,Transient,0.0,0,0,Check-Out,2015-07-01,1
1,Resort Hotel,0,737,2015,July,27,1,0,0,2,0.0,0,BB,PRT,Direct,Direct,0,0,0,C,C,4,No Deposit,NaN,NaN,0,Transient,0.0,0,0,Check-Out,2015-07-01,2
2,Resort Hotel,0,7,2015,July,27,1,0,1,1,0.0,0,BB,GBR,Direct,Direct,0,0,0,A,C,0,No Deposit,NaN,NaN,0,Transient,75.0,0,0,Check-Out,2015-07-02,3
3,Resort Hotel,0,13,2015,July,27,1,0,1,1,0.0,0,BB,GBR,Corporate,Corporate,0,0,0,A,A,0,No Deposit,304.0,NaN,0,Transient,75.0,0,0,Check-Out,2015-07-02,4
4,Resort Hotel,0,14,2015,July,27,1,0,2,2,0.0,0,BB,GBR,Online TA,TA/TO,0,0,0,A,A,0,No Deposit,240.0,NaN,0,Transient,98.0,0,1,Check-Out,2015-07-03,5


In [48]:
# 전체 취소율 베이스라인
N = len(df)
cancel_n = df['is_canceled'].sum()
baseline = cancel_n / N
ci_low, ci_high = proportion_confint(cancel_n, N, alpha=0.05, method='wilson')
print('[전체 취소율 베이스라인]')
print(f'  {baseline*100:.2f}%  (95% CI: {ci_low*100:.2f}% ~ {ci_high*100:.2f}%)')

# 호텔별
print('\n[호텔별 취소율]')
for h, g in df.groupby('hotel'):
    rate = g['is_canceled'].mean()
    lo, hi = proportion_confint(g['is_canceled'].sum(), len(g), alpha=0.05, method='wilson')
    print(f'  {h:14s}  n={len(g):>6,}  취소율={rate*100:5.2f}%  CI=[{lo*100:.2f}, {hi*100:.2f}]')

# 연도별
print('\n[연도별 취소율]')
for y, g in df.groupby('arrival_date_year'):
    rate = g['is_canceled'].mean()
    print(f'  {y}  n={len(g):>6,}  취소율={rate*100:5.2f}%')

[전체 취소율 베이스라인]
  37.04%  (95% CI: 36.77% ~ 37.32%)

[호텔별 취소율]
  City Hotel      n=79,330  취소율=41.73%  CI=[41.38, 42.07]
  Resort Hotel    n=40,060  취소율=27.76%  CI=[27.33, 28.20]

[연도별 취소율]
  2015  n=21,996  취소율=37.02%
  2016  n=56,707  취소율=35.86%
  2017  n=40,687  취소율=38.70%


### EDA 결론

- **전체 취소율 37.04%** — 호텔 산업 평균(보통 20% 내외)보다 상당히 높음
- **City Hotel 41.7% vs Resort Hotel 27.8%** — City 호텔이 14%p 높음. 분석 시 호텔 유형을 통제하거나 분리 검토 필요
- **연도별 변동 폭은 작음** (35.9% ~ 38.7%) — 시계열 트렌드보다는 구조적 요인이 더 큼

> 이후 가설 검증은 전체 데이터로 진행하되, 호텔 유형이 결과를 왜곡할 가능성이 있는 가설은 호텔별로 분리해서도 확인한다.

---

## 2. H1 — lead_time이 길수록 취소율이 높다

예약일에서 체크인일까지의 간격이 멀수록, 그 사이 마음이 바뀌거나 일정이 변경될 가능성이 크다.
실제로 그런가?

### H1 검증 결과 — ✅ **강력하게 지지됨**

| 구간 | n | 취소율 | 95% CI |
|---|---|---|---|
| 0~7일 | 19,746 | **9.6%** | [9.2%, 10.1%] |
| 8~30일 | 18,960 | 27.9% | [27.2%, 28.5%] |
| 31~90일 | 29,553 | 37.7% | [37.2%, 38.3%] |
| 91~180일 | 26,439 | 44.7% | [44.1%, 45.3%] |
| 181~365일 | 21,544 | 55.5% | [54.8%, 56.1%] |
| 365일+ | 3,148 | **67.7%** | [66.0%, 69.3%] |

- **단조 증가** — 구간이 길어질수록 취소율이 일관되게 상승
- **최단 구간(9.6%) vs 최장 구간(67.7%) 차이 +58%p** (95% CI: +56.3%p ~ +59.7%p)
- 카이제곱 p < 0.001, 우연일 가능성 사실상 0

### 기획적 함의

> **"오래전 예약일수록 취소 가능성이 기하급수적으로 커진다"**는 매우 강한 신호.
> 
> - **30일 이내 예약**: 베이스라인보다 낮은 취소율 → 안정적, 추가 정책 불요
> - **90일 이상 예약**: 절반 가까이 취소 → **확정 컨택 / 부분 결제 / 보증금** 등 정책적 개입 가치 높음
> - **365일 이상 예약**: 2/3가 취소 → 이미 "가예약"에 가까움. 보증금 의무화 또는 재확정 마일스톤 설계 필요

---

## 3. H2 — 보증금 없는 예약(No Deposit)이 가장 취소율이 높다

돈을 묶지 않으면 부담 없이 취소한다는 직관적 가설.
실제 데이터는 어떻게 말하는가?

### H2 검증 결과 — ❌ **가설 정반대로 기각, 그러나 더 흥미로운 발견**

| Deposit Type | n | 취소율 | 95% CI |
|---|---|---|---|
| **Non Refund** (환불 불가) | 14,587 | **99.36%** ⚠️ | [99.2%, 99.5%] |
| **No Deposit** (보증금 없음) | 104,641 | 28.38% | [28.1%, 28.7%] |
| **Refundable** (환불 가능) | 162 | 22.22% | [16.5%, 29.2%] |

### 직관과 정반대 — "환불 불가" 예약이 99% 취소된다고?

상식적으로 환불 불가 예약은 **돈을 잃기 때문에** 취소 안 할 것 같은데, 실제로는 거의 100% 취소로 기록되어 있다.

**가능한 해석 (데이터 정의 이슈로 추정)**:
1. **데이터 코딩 차이**: "Non Refund"의 의미가 "고객이 돈을 잃은 채로 안 옴" (= No-Show가 cancel로 기록) 일 가능성
2. **수집 편향**: Non Refund 예약은 결제 이후 시스템상 "cancel" 처리되는 비즈니스 로직이 있을 가능성
3. **상품 구조**: Non Refund 옵션을 선택하는 고객이 특수 세그먼트(예: 도매·재판매 목적)일 가능성

### 기획적 함의 (조건부)

> **이 결과는 그대로 받아들이면 안 되며, 비즈니스 로직 확인이 선행되어야 한다.**
> 
> - 만약 "Non Refund = 결제 후 안 옴 = cancel로 기록" 이라면 → 사실상 매출에는 손실 없음 (이미 결제됨)
> - 만약 "Non Refund 고객도 실제로 환불 받고 취소" 라면 → 정책이 작동하지 않는 것이므로 정책 자체 재검토 필요
> - **권고: 회계·CS 시스템에서 'Non Refund' 처리 흐름을 먼저 확인**

실제 보증금 정책 효과를 측정하려면 **No Deposit vs Refundable 비교**가 더 적절한데, Refundable 표본이 162건뿐이라 신뢰구간이 매우 넓다(±6%p). 통계적으로 차이를 단정하기 어려움.

---

## 4. H3 — 이전 취소 이력자는 재취소 위험이 크다

과거 행동이 미래 행동의 가장 강력한 예측 변수라는 일반론. 실제 데이터로 확인.

### H3 검증 결과 — ✅ **강력하게 지지됨 (단, 단계별 패턴은 비단조)**

#### 이분 비교
| 그룹 | n | 취소율 | 95% CI |
|---|---|---|---|
| 이력 없음 | 112,906 | 33.91% | [33.6%, 34.2%] |
| **이력 있음** | 6,484 | **91.64%** | [90.9%, 92.3%] |
| **차이** | — | **+57.7%p** | [+57.0%p, +58.5%p] |

#### 단계별 (이전 취소 횟수)
| 횟수 | n | 취소율 | 95% CI |
|---|---|---|---|
| 0회 | 112,906 | 33.91% | [33.6%, 34.2%] |
| **1회** | 6,051 | **94.43%** | [93.8%, 95.0%] |
| 2~5회 | 231 | 29.00% | [23.5%, 35.2%] |
| 6회+ | 202 | 79.70% | [73.6%, 84.7%] |

### 핵심 발견

- **"이전 취소 1회만 있어도 재취소 확률 94%"** — 단일 변수로 이만큼 강한 신호는 드물다
- 2~5회 구간이 29%로 급락하는 건 **표본 크기 231건의 작은 n** 때문일 가능성 큼 (CI 폭이 넓음 [23.5%, 35.2%])
- 6회+ 구간(79.7%)도 표본 202건으로 신뢰구간이 넓어, 단계별 단조성을 단정하기 어려움

### 기획적 함의

> **"이전 취소 1회 이상" 단순 플래그만으로도 매우 강력한 위험 신호.**
> 
> - 신규 예약 시점에 고객 ID로 이력 자동 조회 → "이력 있음" 플래그 부여
> - 이력 보유자에게는: **(a) 보증금 의무화** 또는 **(b) 결제 선납** 또는 **(c) 사전 확정 컨택 강화**
> - 단, **상습 취소자(2회+)는 표본이 작아** 추가 데이터 수집 후 정밀 정책 설계 필요

---

## 5. H4 — 마켓 세그먼트별 취소율 차이

예약이 어느 채널을 통해 들어왔는지가 취소율에 영향을 주는가? 채널별 마케팅·운영 차별화의 근거를 찾는다.

### H4 검증 결과 — ✅ **세그먼트별 차이 매우 큼**

| 세그먼트 | n | 취소율 | 95% CI |
|---|---|---|---|
| **Groups** ⚠️ | 19,811 | **61.06%** | [60.4%, 61.7%] |
| Online TA | 56,477 | 36.72% | [36.3%, 37.1%] |
| Offline TA/TO | 24,219 | 34.32% | [33.7%, 34.9%] |
| Aviation | 237 | 21.94% | [17.1%, 27.6%] |
| Corporate | 5,295 | 18.73% | [17.7%, 19.8%] |
| **Direct** ✅ | 12,606 | **15.34%** | [14.7%, 16.0%] |
| Complementary | 743 | 13.06% | [10.8%, 15.7%] |

*(Undefined 2건은 시각화에서 제외)*

### 핵심 발견

- **Groups 세그먼트가 가장 위험** — 취소율 61%, 거의 평균의 2배
- **Direct 채널이 가장 안전** — 취소율 15%, 평균의 절반 이하
- **Online TA(47% 비중, 최대 채널)는 평균 수준** — 예약량 자체는 많지만 취소율도 평균과 유사

### 기획적 함의

> **채널별로 차별화된 정책·운영이 정량적으로 정당화됨**
> 
> - **Groups**: 단체 예약 특성상 일정 변경·리더 결정에 따른 취소가 많음 → 
>   - 단체 담당자 전담 컨택 채널 운영
>   - 단체 예약 보증금 정책 강화 (사전 부분 결제)
>   - 인원 변동 시 부분 취소 가능 옵션 (전부 취소 방지)
> - **Direct**: 가장 안전한 채널 → 마케팅 투자 확대, 자체 브랜드 채널 강화
> - **Online TA**: 매출 비중 절대 1위지만 취소율도 평균 → 
>   - 채널 수수료 vs 취소 손실의 ROI 재평가
>   - 채널 내 분기 (예: 즉시 결제 / 대기 결제) 전환율 별도 측정

---

## 6. H5 — 예약 변경 횟수와 취소율 관계

변경이 잦은 예약은 "확정되지 못한 예약"이라 결국 취소될 가능성이 클 것이다. 라는 직관적 가설.

### H5 검증 결과 — ❌ **가설 정반대 — 변경 안 한 예약이 더 많이 취소됨**

| 변경 횟수 | n | 취소율 | 95% CI |
|---|---|---|---|
| **0회** | 101,314 | **40.85%** | [40.5%, 41.2%] |
| 1회 | 12,701 | 14.23% | [13.6%, 14.9%] |
| 2회 | 3,805 | 20.13% | [18.9%, 21.4%] |
| 3회+ | 1,570 | 16.56% | [14.8%, 18.5%] |

### 직관과 정반대 — 왜?

변경이 "위험 신호"가 아니라 **"예약을 적극 관리하고 있다는 의지의 신호"**로 해석됨.

- **0회 변경 = 무관심** → 마음이 떠나면 그냥 취소
- **1회+ 변경 = 능동적 관여** → 일정·인원 조정해서라도 가려는 의지

변경 1회 그룹이 14%로 가장 낮고, 2회·3회+ 그룹은 다시 약간 올라가는 패턴 → "적당한 변경은 헌신의 신호, 과한 변경은 불안정의 신호" 라는 미세한 해석도 가능 (단, 차이 크지 않음).

### 기획적 함의 — 데이터가 가설을 뒤집음

> **"예약 변경 = 위험 신호" 라는 통념을 데이터로 반박할 수 있는 사례.**
> 
> - **0회 변경 + 임박 예약**: 진짜 위험군 → **사전 확정 컨택 트리거 대상**
> - **변경 발생 예약**: 오히려 안전. 별도 페널티나 추가 마찰을 주면 안 됨 (긍정 행동을 처벌하는 셈)
> - **운영 시스템 권고**: 변경 횟수를 "문제 신호"로 분류하던 규칙이 있다면 재검토 필요

---

## 7. H6 — 월별(시즌별) 취소율 차이

### H6 검증 결과 — ✅ **차이는 있으나 효과 크기는 작음**

- **최저: 11월 (31.2%)** / **최고: 6월 (41.5%)** — 약 10%p 폭의 변동
- **취소율 상승기**: 4~6월 (40~41%), 9~10월 (38~39%) — 휴가/여행 성수기 시작 직전
- **취소율 안정기**: 11~3월 (31~35%) — 비수기, 예약 자체가 신중
- 카이제곱 p < 0.001 (차이는 통계적으로 유의)

### 기획적 함의

> **시즌성은 부차적 변수 — 다른 가설(H1, H3, H4)에 비해 효과 크기 작음**
> 
> - 4~9월 (취소율 상승기): 오버부킹 정책 보수적 운영
> - 11~2월 (안정기): 프로모션 / 장기 예약 유치 캠페인
> - 단, 이 패턴은 **H1(lead time)·H4(채널) 효과와 혼재**되어 있을 수 있음 (예: 성수기엔 장기 예약 비중이 큼) → 인과 분석 시 통제 필요

---

## 8. 종합 인사이트

6개 가설을 효과 크기와 신뢰도, 실행가능성 기준으로 다시 정리한다.

### 가설 검증 결과 요약

| 가설 | 결과 | 효과 크기 | 신뢰도 | 비고 |
|---|---|---|---|---|
| **H3** 이전 취소 이력 | ✅ 강력 지지 | **+57.7%p** | 매우 높음 | **단일 변수 최강 신호** |
| **H1** Lead time | ✅ 강력 지지 | +58%p (극단 비교) | 매우 높음 | 단조 증가, 강한 패턴 |
| **H4** 마켓 세그먼트 | ✅ 지지 | Groups 61% vs Direct 15% | 높음 | 채널별 차별화 근거 명확 |
| **H5** Booking changes | ❌ **반대 결과** | 0회 41% vs 1회+ ~15% | 높음 | 변경 = 의지 신호, 통념 반전 |
| **H6** 시즌성 | ✅ 지지 (약함) | 약 10%p 변동 | 중간 | 다른 변수와 혼재 가능 |
| **H2** Deposit type | ⚠️ **데이터 정의 의심** | Non Refund 99% (?) | 낮음 | 비즈니스 로직 확인 선행 필요 |

### 고위험 예약자 프로필 (다중 신호 결합 시)

다음 조건을 동시에 만족하는 예약은 취소 위험이 **압도적으로 높음**:

1. **이전 취소 이력 있음** (단일 신호로 91% 취소)
2. **체크인 90일+ 전에 예약** (취소율 45~68%)
3. **Groups 세그먼트** (취소율 61%)
4. **변경 이력 없음** (취소율 41%)

→ 이 4개 신호 모두 만족하는 예약은 **거의 확실하게 취소될 후보군**으로 운영적 핸들링이 필요.

### 저위험 예약자 프로필

1. **30일 이내 예약 (특히 7일 이내)** (취소율 9.6%)
2. **Direct / Corporate 채널** (취소율 15~19%)
3. **이전 취소 이력 없음**
4. **변경 이력 1회+** (취소율 14%)

→ 이 그룹은 **별도 정책 개입 불요**, 안정적 운영 가능.

---

## 9. 기획 권고 — 정책 / UX / 운영

분석에서 도출된 신호들을 실제 적용 가능한 액션으로 변환한다.

### 우선순위 1 — 즉시 적용 가능 (Quick Win)

#### 🎯 R1. "이전 취소 이력자" 자동 식별 + 위험 알림
- **근거**: 이력 1회만 있어도 재취소 확률 91~94% (n=6,484, CI 매우 좁음)
- **액션**:
  - 신규 예약 등록 시 고객 ID로 이력 자동 조회
  - "위험 예약" 플래그 부여 → 운영진에 알림 / 보증금 의무화 / 사전 확정 컨택
- **기대 임팩트**: 위험군 6,484건 중 91%가 취소되므로, 이 그룹을 사전 핸들링하면 **연간 약 5,900건의 취소 이벤트를 줄일 잠재력**

#### 🎯 R2. "변경 0회 + 장기 예약" 사전 확정 컨택 트리거
- **근거**: 변경 0회 그룹 취소율 41% (101,314건), lead time 90일+ 그룹 45~68%
- **액션**:
  - 체크인 7일 / 3일 전 자동 컨택 (이메일·SMS·앱 푸시)
  - 컨택 응답 없으면 "위험" 분류 → 객실 재배치·오버부킹 풀에 포함
- **기대 임팩트**: 컨택 응답률에 따라 5~15%p 취소율 감소 가능 (별도 A/B 검증 필요)

---

### 우선순위 2 — 정책 재설계

#### 🎯 R3. 채널별 차별화 정책
- **Groups 세그먼트** (취소율 61%, 가장 위험):
  - 단체 담당자 전담 컨택 운영
  - 단체 예약 보증금 정책 강화 (인원당 부분 결제)
  - 부분 취소 옵션 도입 (전부 취소 방지)
- **Direct 채널** (취소율 15%, 가장 안전):
  - 마케팅 투자 확대 — ROI 가장 높음
  - Direct 채널 사용 인센티브 (포인트·할인)
- **Online TA** (매출 1위, 평균 취소율):
  - 채널 수수료 vs 취소 손실 비교 → 채널 ROI 재평가
  - TA 내 "즉시 결제 vs 후불" 옵션 분리 후 전환율 별도 측정

#### 🎯 R4. Lead Time 기반 단계적 결제 정책
| Lead Time | 현재 정책 | 권고 정책 |
|---|---|---|
| 30일 이내 | 보증금 없음 | 유지 (안정적) |
| 31~90일 | 보증금 없음 | 유지하되 1회 확정 컨택 |
| 91~180일 | 보증금 없음 | **15~30% 부분 결제 도입** |
| 181일+ | 보증금 없음 | **50% 보증금 또는 "가예약" 분리 트랙** |

---

### 우선순위 3 — 운영 시스템 개선

#### 🎯 R5. "Booking Changes" 처리 로직 재검토
- **근거**: 변경 1회+ 그룹 취소율 14% (변경 0회 그룹 41%보다 낮음)
- **현재 통념(추정)**: "변경이 잦은 예약 = 위험" → 페널티·마찰 부여
- **권고**: "변경 = 헌신 신호"로 재해석 → 변경 시 마찰 최소화 (간편 변경 UI), 페널티 제거
- **주의**: 단, 이 결과는 관측 데이터 기반이라 인과 검증 필요 (변경 자체가 취소를 막는 게 아니라, 변경할 만큼 의지있는 사람이 안 취소한다는 해석도 가능)

#### 🎯 R6. 시즌별 오버부킹 전략 차등화
- **취소율 상승기 (4~9월)**: 오버부킹 비율 보수적 (취소율 + 신뢰구간 상한 기준)
- **안정기 (11~2월)**: 오버부킹 적극적 + 장기 예약 프로모션

---

### 권고 적용 전 추가로 필요한 검증

| 항목 | 이유 |
|---|---|
| **1. Non Refund 데이터 정의 확인** | 99% 취소율의 비즈니스 의미 확인 (회계·시스템 처리 흐름) |
| **2. 중복 26.8%의 본질** | 단순 중복인지, 동일 조건 다중 행인지 데이터 명세 확인 |
| **3. R1 (이력자 정책) A/B 검증** | 보증금 의무화 시 예약 자체가 줄어들 가능성 정량화 |
| **4. R5 (변경 로직) 인과 검증** | 변경 권장이 실제로 취소율을 낮추는지, 단순 상관일 뿐인지 |
| **5. 한국 시장 일반화** | 본 데이터는 포르투갈 호텔. 한국 호텔/숙박 시장에서 동일 패턴인지 별도 검증 |

---

## 결론

본 분석은 약 12만 건의 호텔 예약 데이터를 6개 가설로 검증해 **취소율의 강력한 예측 신호 3개**(이전 취소 이력 / Lead Time / 마켓 세그먼트)와 **통념을 반박하는 1개 발견**(변경 횟수)을 도출했다.

특히 **"이전 취소 이력 1회 = 재취소 91%"**는 단일 변수로는 매우 드물게 강한 신호로, 즉시 적용 가능한 운영 정책의 근거가 된다.

분석 과정에서 **데이터 정의 이슈(Non Refund 99% 취소율)**도 발견했는데, 이는 기획자가 데이터를 다룰 때 "숫자 그대로 받아들이지 않고 비즈니스 로직과 교차 검증"해야 한다는 사례가 된다.

> **다음 단계**: 위 권고 중 **R1 (이력자 자동 식별)**을 우선 도입하고, 6개월 후 취소율 변화를 사전·사후 비교(또는 통제군 A/B)로 검증한다.